# Tarea 1 Parte B
Integrantes:
- Carlos Raúl Sánchez Figueroa
- Ulises Omar Montes Correa
- Diego Córdoba Gómez
- Guillermo Mateos Collado

In [0]:
# Librerías usadas para manipulación y transformación de los datos

import pandas as pd          # manejo auxiliar de datos
import re                    # expresiones regulares
from pyspark.sql import functions as F  # funciones generales de Spark
from pyspark.sql.functions import expr, size, col
from pyspark.sql.functions import to_timestamp, hour  # manejo de fechas y particiones por hora
from pyspark.sql.functions import schema_of_json      # inferir esquema cuando hay datos en json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, ArrayType

In [0]:
%sql
create catalog if not exists dev;

In [0]:
%sql
create database if not exists dev.ciencias_data

In [0]:
%sql
create volume if not exists dev.ciencias_data.session_data;

## Parte 1
#### Cree una tabla bronce en formato delta y particionado por hora para session part1.csv.

In [0]:
# lectura del archivo de sesiones en formato csv
# el archivo usa "|" como separador y contiene encabezados
df = spark.read.format("csv") \
    .option("sep","|") \
    .option("header","true") \
    .load("/Volumes/dev/ciencias_data/session_data/sessions_part1.csv")

# se agregan columnas de control para registrar el momento de carga
# y el origen de los datos (sniffer Arkime)
df = df.withColumn("_load_timestamp", F.lit(F.current_timestamp())) \
       .withColumn("_source", F.lit("Arkime"))

# el campo timestamp se convierte a formato de fecha
# a partir de él se extrae la hora, que se usará como partición
df = df.withColumn(
    "event_timestamp",
    to_timestamp("timestamp")
).withColumn(
    "hour_partition",
    hour("event_timestamp")
)

# escritura de los datos en formato Delta
# la tabla corresponde a la capa bronze y se particiona por hora
df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("hour_partition") \
  .saveAsTable("dev.ciencias_data.bronze_sessions")# Leemos el csv y cargamos
df=spark.read.format("csv").option("sep","|").option("header","true").load("/Volumes/dev/ciencias_data/session_data/sessions_part1.csv")

df=df.withColumn("_load_timestamp",F.lit(F.current_timestamp())).withColumn("_source",F.lit("Arkime"))
df = df.withColumn(
    "event_timestamp",
    to_timestamp("timestamp")
).withColumn(
    "hour_partition",
    hour("event_timestamp")
)

df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("hour_partition") \
  .saveAsTable("dev.ciencias_data.bronze_sessions")

In [0]:
%sql
DROP TABLE IF EXISTS dev.ciencias_data.silver_sessions;

#### 2. Aplicamos las transformaciones al dataset

In [0]:
def to_snake_case(name):
    """Convierte cadenas de formato camelCase a snake_case"""
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

# Lectura de la tabla bronze
df_bronze = spark.table("dev.ciencias_data.bronze_sessions")

# Definición explícita del esquema mapeando las columnas finales a sus claves JSON originales
esquema_json = StructType([
    StructField("communityId", StringType(), True),
    StructField("dstASN", StringType(), True),
    StructField("dstBytes", LongType(), True),
    StructField("dstDataBytes", LongType(), True),
    StructField("dstGEO", StringType(), True),
    StructField("dstIp", StringType(), True),
    StructField("dstMac", ArrayType(StringType()), True),
    StructField("dstMacCnt", IntegerType(), True),
    StructField("dstPackets", LongType(), True),
    StructField("dstPort", IntegerType(), True),
    StructField("fileId", ArrayType(StringType()), True),
    StructField("firstPacket", LongType(), True),
    StructField("initRtt", IntegerType(), True),
    StructField("ipProtocol", IntegerType(), True),
    StructField("lastPacket", LongType(), True),
    StructField("length", LongType(), True),
    StructField("node", StringType(), True),
    StructField("protocol", ArrayType(StringType()), True),
    StructField("protocolCnt", IntegerType(), True),
    StructField("segmentCnt", IntegerType(), True),
    StructField("srcAsn", StringType(), True),
    StructField("srcBytes", LongType(), True),
    StructField("srcDataBytes", LongType(), True),
    StructField("srcGEO", StringType(), True),
    StructField("srcIp", StringType(), True),
    StructField("srcMac", ArrayType(StringType()), True),
    StructField("srcMacCnt", IntegerType(), True),
    StructField("srcPackets", LongType(), True),
    StructField("srcPayload8", StringType(), True),
    StructField("srcPort", IntegerType(), True),
    StructField("timestamp", LongType(), True),
    StructField("totBytes", LongType(), True),
    StructField("totDataBytes", LongType(), True),
    StructField("totPackets", LongType(), True),
    StructField("packetLen", ArrayType(IntegerType()), True),
    StructField("packetPos", ArrayType(IntegerType()), True),
    StructField("cert", ArrayType(StringType()), True)
    ])

# Se parsea el json completo utilizando el esquema explícito
df_parsed = df_bronze.withColumn(
    "json_data",
    F.from_json(F.col("data"), esquema_json)
).select(
    "json_data.*",
    "event_timestamp", "_load_timestamp", "_source", "hour_partition"
)

# Transformaciones de métricas resumidas y eliminación de columnas procesadas
df_silver = df_parsed.withColumn(
    "packet_len_sum",
    expr("aggregate(packetLen, cast(0 as bigint), (acc, x) -> acc + cast(x as bigint))")
).withColumn(
    "packet_len_avg",
    col("packet_len_sum") / size(col("packetLen"))
).withColumn(
    "packet_len_min",
    expr("array_min(packetLen)")
).withColumn(
    "packet_len_max",
    expr("array_max(packetLen)")
).drop("packetLen", "packetPos", "cert")

# Conversión de tiempos en milisegundos a timestamp
df_silver = df_silver.withColumn(
    "firstPacket", (col("firstPacket") / 1000).cast("timestamp")
).withColumn(
    "lastPacket", (col("lastPacket") / 1000).cast("timestamp")
).withColumn(
    "timestamp", (col("timestamp") / 1000).cast("timestamp")
)

# Estandarización de nombres a snake_case
for column in df_silver.columns:
    df_silver = df_silver.withColumnRenamed(column, to_snake_case(column))

# Escritura de la tabla silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.ciencias_data.silver_sessions")

df_silver.display()

## Diccionario de datos

- community_id: identificador único de la sesión
- src_ip / dst_ip: direcciones IP de origen y destino
- src_port / dst_port: puertos de origen y destino
- src_geo / dst_geo: país asociado a la IP
- src_mac / dst_mac: direcciones MAC observadas en la sesión
- src_bytes / dst_bytes: bytes enviados desde origen y hacia destino
- src_data_bytes / dst_data_bytes: bytes de datos útiles
- src_packets / dst_packets: número de paquetes

- tot_bytes / tot_data_bytes / tot_packets: totales de la sesión
- first_packet / last_packet: tiempo del primer y último paquete
- timestamp / event_timestamp: marca de tiempo del evento
- load_timestamp: momento en que el dato fue cargado
- source: origen de los datos

- protocol / ip_protocol / protocol_cnt: información de protocolos
- dst_asn: sistema autónomo asociado a la IP destino
- node: nodo donde se capturó la sesión
- segment_cnt: número de segmentos

- packet_len_sum / packet_len_avg / packet_len_min / packet_len_max: estadísticas de longitud de paquetes

### Parte 3
####Hacemos una carga incremental de session part2.csv, a su vez aplicando las mismas transformaciones que el dataset anterior

In [0]:
# Lectura del segundo archivo
df2 = spark.read.format("csv") \
    .option("sep", "|") \
    .option("header", "true") \
    .load("/Volumes/dev/ciencias_data/session_data/sessions_part2.csv")

# Asignación de metadatos de carga
df2 = df2.withColumn("_load_timestamp", F.lit(F.current_timestamp())) \
         .withColumn("_source", F.lit("Arkime"))

# Conversión de timestamp y partición temporal
df2 = df2.withColumn(
    "event_timestamp",
    to_timestamp("timestamp")
).withColumn(
    "hour_partition",
    hour("event_timestamp")
)

# Parseo del json estructurado con el esquema explícito definido en la Parte 2
df_parsed_inc = df2.withColumn(
    "json_data",
    F.from_json(F.col("data"), esquema_json)
).select(
    "json_data.*",
    "event_timestamp", "_load_timestamp", "_source", "hour_partition"
)

# Transformaciones de métricas en la carga incremental
df_silver_inc = df_parsed_inc.withColumn(
    "packet_len_sum",
    expr("aggregate(packetLen, cast(0 as bigint), (acc, x) -> acc + cast(x as bigint))")
).withColumn(
    "packet_len_avg",
    col("packet_len_sum") / size(col("packetLen"))
).withColumn(
    "packet_len_min",
    expr("array_min(packetLen)")
).withColumn(
    "packet_len_max",
    expr("array_max(packetLen)")
).drop("packetLen", "packetPos", "cert")

# Conversión de los campos de tiempo
df_silver_inc = df_silver_inc.withColumn(
    "firstPacket", (col("firstPacket") / 1000).cast("timestamp")
).withColumn(
    "lastPacket", (col("lastPacket") / 1000).cast("timestamp")
).withColumn(
    "timestamp", (col("timestamp") / 1000).cast("timestamp")
)

# Estandarización a snake_case
for column in df_silver_inc.columns:
    df_silver_inc = df_silver_inc.withColumnRenamed(column, to_snake_case(column))

# Escritura en formato Delta modo append para anexar las sesiones nuevas
df_silver_inc.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dev.ciencias_data.silver_sessions")
df_silver_inc.limit(5).display()

# Parte 4

In [0]:
# lectura de la tabla silver para realizar consultas y análisis
df = spark.table("dev.ciencias_data.silver_sessions")

# visualización rápida del contenido
df.limit(5).display()

## Número de sesiones por país de origen

In [0]:
# conteo de sesiones por país de origen (src_geo)
# ordenado de mayor a menor número de sesiones
df.groupBy("src_geo") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## Número de sesiones por país destino

In [0]:
# conteo de sesiones por país destino (dst_geo), ordenado de mayor a menor
df.groupBy("dst_geo") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## Número de sesiones por src_ip y dst_ip

In [0]:
# conteo de sesiones por combinación de IP origen y destino
df.groupBy("src_ip", "dst_ip") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## tot_bytes, tot_data_bytes y tot_packets por src_ip y protocol

In [0]:
# tráfico total generado por cada IP origen y protocolo
# se suman bytes, bytes de datos y número de paquetes
df.groupBy("src_ip", "protocol") \
    .agg(
        F.sum("tot_bytes").alias("sum_tot_bytes"),
        F.sum("tot_data_bytes").alias("sum_tot_data_bytes"),
        F.sum("tot_packets").alias("sum_tot_packets")
    ) \
    .orderBy(F.desc("sum_tot_bytes")) \
    .show()

## tot_bytes, tot_data_bytes y tot_packets por src_mac y dst_mac

In [0]:
# análisis del tráfico por dirección MAC de origen
# src_mac es un arreglo, por lo que primero se separa cada valor con explode
df_src_mac = df.withColumn("src_mac_exploded", F.explode("src_mac"))

# agregación del tráfico total asociado a cada MAC origen
df_src_mac.groupBy("src_mac_exploded") \
    .agg(
        F.sum("tot_bytes").alias("sum_tot_bytes"),
        F.sum("tot_data_bytes").alias("sum_tot_data_bytes"),
        F.sum("tot_packets").alias("sum_tot_packets")
    ) \
    .orderBy(F.desc("sum_tot_bytes")) \
    .show()

In [0]:
# análisis del tráfico por dirección MAC de destino
# dst_mac también es un arreglo, por lo que se separa cada valor con explode
df_dst_mac = df.withColumn("dst_mac_exploded", F.explode("dst_mac"))

# agregación del tráfico total asociado a cada MAC destino
df_dst_mac.groupBy("dst_mac_exploded") \
    .agg(
        F.sum("tot_bytes").alias("sum_tot_bytes"),
        F.sum("tot_data_bytes").alias("sum_tot_data_bytes"),
        F.sum("tot_packets").alias("sum_tot_packets")
    ) \
    .orderBy(F.desc("sum_tot_bytes")) \
    .show()

## Mínimo, máximo y promedio por src_ip, srcIp, dstIp, srcMac y dstMac

In [0]:
# estadísticas básicas del tráfico por IP de origen
# se calculan valores mínimo, máximo y promedio para bytes, data_bytes y paquetes
df.groupBy("src_ip") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

In [0]:
# estadísticas del tráfico agrupadas por IP destino
# se calculan valores mínimo, máximo y promedio para bytes totales,
# bytes de datos y número de paquetes asociados a cada destino
df.groupBy("dst_ip") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

In [0]:
# estadísticas de tráfico agrupadas por dirección MAC de origen
# se calculan valores mínimo, máximo y promedio para bytes totales,
# bytes de datos y número de paquetes asociados a cada MAC
df_src_mac.groupBy("src_mac_exploded") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

In [0]:
# estadísticas de tráfico agrupadas por dirección MAC de destino
# se calculan valores mínimo, máximo y promedio para bytes totales,
# bytes de datos y número de paquetes asociados a cada MAC destino
df_dst_mac.groupBy("dst_mac_exploded") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

## Top 5 src_ip y src_mac con más sesiones

In [0]:
# identificación de las 5 IP de origen con más sesiones registradas
df.groupBy("src_ip") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(5) \
    .show()

In [0]:
# top 5 direcciones MAC de origen con mayor número de sesiones
df_src_mac.groupBy("src_mac_exploded") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(5) \
    .show()

## Número de src_mac y dst_mac por sesión

In [0]:
# número de direcciones MAC de origen y destino asociadas a cada sesión
# se usa size() para obtener el tamaño del arreglo en cada campo
df.select(
    "src_ip",
    F.size("src_mac").alias("num_src_mac"),
    F.size("dst_mac").alias("num_dst_mac")
).show()

## Protocolos más usados 
Para esto usaremos la columna Protocol

In [0]:
# número de sesiones agrupadas por protocolo de red
# permite identificar cuáles protocolos se utilizan con mayor frecuencia
df.groupBy("protocol") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## Páginas web más visitadas

In [0]:
# número de sesiones agrupadas por ASN de destino
# permite identificar las redes destino más frecuentes en el tráfico
df.groupBy("dst_asn") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()